In [6]:
import re
import pandas as pd
from pathlib import Path

# ---------- CONFIG ----------
INPUT_PATH = Path(r"D:\python\Projects\Webloganalyzer\data\raw\finalweblog.txt")
OUTPUT_PATH = Path("D:\python\Projects\Webloganalyzer\data\processed\parsed_logs.csv")

# ---------- REGEX PATTERN ----------
log_pattern = re.compile(
    r'(\S+) (\S+) (\S+) \[(.*?)\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+)'
)
print("Using file:", INPUT_PATH)
print("File exists:", INPUT_PATH.exists())

# ---------- PARSER FUNCTION ----------
def parse_log(file_path):
    rows = []
    failed = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            match = log_pattern.match(line)

            if match:
                ip, ident, authuser, timestamp, method, url, protocol, status, size = match.groups()

                # Handle missing size
                size = int(size) if size != '-' else 0

                rows.append({
                    "ip": ip,
                    "ident": ident,
                    "authuser": authuser,
                    "timestamp": timestamp,
                    "method": method,
                    "url": url,
                    "protocol": protocol,
                    "status": int(status),
                    "size": size
                })
            else:
                failed += 1

    df = pd.DataFrame(rows)

    print(f"✅ Parsed rows: {len(df)}")
    print(f"⚠️ Failed rows: {failed}")

    return df


# ---------- CLEANING + FEATURE ENGINEERING ----------
def process_dataframe(df):
    # Convert timestamp
    df['timestamp'] = pd.to_datetime(
        df['timestamp'],
        format='%d/%b/%Y:%H:%M:%S %z',
        errors='coerce'
    )

    # Drop invalid timestamps
    df = df.dropna(subset=['timestamp'])

    # Feature Engineering
    df['hour'] = df['timestamp'].dt.hour
    df['day'] = df['timestamp'].dt.day
    df['weekday'] = df['timestamp'].dt.weekday

    df['url_depth'] = df['url'].apply(lambda x: x.count('/'))
    df['is_error'] = (df['status'] >= 400).astype(int)

    return df


# ---------- MAIN PIPELINE ----------
def main():
    print("🚀 Starting log parsing...")

    df = parse_log(INPUT_PATH)

    print("🧹 Cleaning & processing...")
    df = process_dataframe(df)

    # Ensure output folder exists
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(OUTPUT_PATH, index=False)

    print(f"💾 Saved to: {OUTPUT_PATH}")
    print("✅ Done!")


if __name__ == "__main__":
    main()

<>:7: SyntaxWarning: invalid escape sequence '\p'
<>:7: SyntaxWarning: invalid escape sequence '\p'
C:\Users\DELL\AppData\Local\Temp\ipykernel_21580\1538267761.py:7: SyntaxWarning: invalid escape sequence '\p'
  OUTPUT_PATH = Path("D:\python\Projects\Webloganalyzer\data\processed\parsed_logs.csv")


Using file: D:\python\Projects\Webloganalyzer\data\raw\finalweblog.txt
File exists: True
🚀 Starting log parsing...
✅ Parsed rows: 99719
⚠️ Failed rows: 281
🧹 Cleaning & processing...
💾 Saved to: D:\python\Projects\Webloganalyzer\data\processed\parsed_logs.csv
✅ Done!


In [7]:
import pandas as pd
from pathlib import Path

# ---------- PATHS ----------
BASE_DIR = Path(r"D:\python\Projects\Webloganalyzer")

INPUT_PATH = BASE_DIR / "data/processed/parsed_logs.csv"
OUTPUT_PATH = BASE_DIR / "data/processed/clean_logs.csv"


def clean_data():
    print("📂 Loading parsed data...")
    df = pd.read_csv(INPUT_PATH)

    print("Rows before cleaning:", len(df))

    # -------------------------
    # FIX DATA TYPES
    # -------------------------
    df['status'] = pd.to_numeric(df['status'], errors='coerce')
    df['size'] = pd.to_numeric(df['size'], errors='coerce')

    # Drop invalid rows
    df = df.dropna(subset=['status', 'size'])

    df['status'] = df['status'].astype(int)
    df['size'] = df['size'].astype(int)

    # -------------------------
    # VALIDATE PROTOCOL
    # -------------------------
    df['protocol'] = df['protocol'].str.extract(r'(HTTP/\d\.\d)', expand=False)
    df = df.dropna(subset=['protocol'])

    # -------------------------
    # HANDLE MISSING VALUES
    # -------------------------
    df['method'] = df['method'].fillna("UNKNOWN")
    df['url'] = df['url'].fillna("/")

    # -------------------------
    # REMOVE DUPLICATES
    # -------------------------
    df = df.drop_duplicates()

    # -------------------------
    # BASIC FEATURE CHECK
    # (already created in parser, just ensure exist)
    # -------------------------
    required_cols = ['hour', 'day', 'weekday', 'url_depth', 'is_error']
    for col in required_cols:
        if col not in df.columns:
            print(f"⚠️ Missing column: {col}")

    print("Rows after cleaning:", len(df))

    # -------------------------
    # SAVE CLEAN DATA
    # -------------------------
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)

    print(f"💾 Saved clean data → {OUTPUT_PATH}")


if __name__ == "__main__":
    clean_data()

📂 Loading parsed data...
Rows before cleaning: 99719
Rows after cleaning: 99691
💾 Saved clean data → D:\python\Projects\Webloganalyzer\data\processed\clean_logs.csv


In [8]:
import pandas as pd
from pathlib import Path
import hashlib

# ---------- PATHS ----------
BASE_DIR = Path(r"D:\python\Projects\Webloganalyzer")

INPUT_PATH = BASE_DIR / "data/processed/clean_logs.csv"
OUTPUT_PATH = BASE_DIR / "data/processed/final_features.csv"


def feature_engineering():
    print("📂 Loading clean data...")
    df = pd.read_csv(INPUT_PATH)

    print("Rows before feature engineering:", len(df))

    # -----------------------------
    # URL FEATURES (VERY IMPORTANT)
    # -----------------------------
    df['url_len'] = df['url'].str.len()
    df['url_depth'] = df['url'].str.count('/')

    df['is_image'] = df['url'].str.contains(
        r'\.(gif|jpg|png)', case=False, regex=True
    ).astype(int)

    df['is_html'] = df['url'].str.contains(
        r'\.html', case=False, regex=True
    ).astype(int)

    # -----------------------------
    # METHOD & PROTOCOL ENCODING
    # -----------------------------
    df = pd.get_dummies(df, columns=['method', 'protocol'], drop_first=True)

    # -----------------------------
    # IP ENCODING (HASH BASED)
    # -----------------------------
    df['ip_enc'] = df['ip'].apply(
        lambda x: int(hashlib.md5(x.encode()).hexdigest(), 16) % 10000
    )

    # -----------------------------
    # DROP UNUSED COLUMNS
    # -----------------------------
    df.drop(columns=[
        'ip',
        'url',
        'ident',
        'authuser',
        'timestamp'  # keep if you want time-based models later
    ], inplace=True, errors='ignore')

    # -----------------------------
    # FINAL CHECK
    # -----------------------------
    print("Final columns:", df.columns.tolist())
    print("\nSample:")
    print(df.head())

    # -----------------------------
    # SAVE FINAL DATASET
    # -----------------------------
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)

    print(f"\n💾 Saved → {OUTPUT_PATH}")


if __name__ == "__main__":
    feature_engineering()

📂 Loading clean data...
Rows before feature engineering: 99691


C:\Users\DELL\AppData\Local\Temp\ipykernel_21580\1250736790.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_image'] = df['url'].str.contains(


Final columns: ['status', 'size', 'hour', 'day', 'weekday', 'url_depth', 'is_error', 'url_len', 'is_image', 'is_html', 'method_HEAD', 'method_POST', 'ip_enc']

Sample:
   status  size  hour  day  weekday  url_depth  is_error  url_len  is_image  \
0     200  6245     0    1        5          3         0       16         0   
1     200  3985     0    1        5          3         0       19         0   
2     200  4085     0    1        5          4         0       44         0   
3     304     0     0    1        5          3         0       31         0   
4     200  4179     0    1        5          4         0       47         1   

   is_html  method_HEAD  method_POST  ip_enc  
0        0        False        False    4029  
1        0        False        False    8049  
2        1        False        False    3990  
3        1        False        False    1699  
4        0        False        False    3990  

💾 Saved → D:\python\Projects\Webloganalyzer\data\processed\final_features.

In [9]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# ---------- PATHS ----------
BASE_DIR = Path(r"D:\python\Projects\Webloganalyzer")

INPUT_PATH = BASE_DIR / "data/processed/final_features.csv"
MODEL_PATH = BASE_DIR / "models/error_model.pkl"


def train_model():
    print("📂 Loading dataset...")
    df = pd.read_csv(INPUT_PATH)

    print("Total rows:", len(df))

    # ---------------------------
    # TARGET & FEATURES
    # ---------------------------
    y = df["is_error"]

    # Remove leakage
    X = df.drop(columns=["is_error", "status"], errors='ignore')

    print("Features used:", list(X.columns))

    # ---------------------------
    # TRAIN TEST SPLIT
    # ---------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # ---------------------------
    # MODEL
    # ---------------------------
    model = RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    )

    print("\n🚀 Training model...")
    model.fit(X_train, y_train)

    # ---------------------------
    # PREDICTION
    # ---------------------------
    preds = model.predict(X_test)

    # ---------------------------
    # EVALUATION
    # ---------------------------
    print("\n📊 Model Performance")

    print("\nAccuracy:", accuracy_score(y_test, preds))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))

    print("\nClassification Report:")
    print(classification_report(y_test, preds))

    # ---------------------------
    # FEATURE IMPORTANCE (VERY IMPORTANT)
    # ---------------------------
    print("\n🔥 Top Features:")
    importances = pd.Series(model.feature_importances_, index=X.columns)
    print(importances.sort_values(ascending=False).head(10))

    # ---------------------------
    # SAVE MODEL
    # ---------------------------
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

    bundle = {
        "model": model,
        "features": list(X.columns)
    }

    joblib.dump(bundle, MODEL_PATH)

    print(f"\n💾 Model saved → {MODEL_PATH}")


if __name__ == "__main__":
    train_model()

📂 Loading dataset...
Total rows: 99691
Features used: ['size', 'hour', 'day', 'weekday', 'url_depth', 'url_len', 'is_image', 'is_html', 'method_HEAD', 'method_POST', 'ip_enc']

🚀 Training model...

📊 Model Performance

Accuracy: 0.9985957169366568

Confusion Matrix:
[[19834     4]
 [   24    77]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19838
           1       0.95      0.76      0.85       101

    accuracy                           1.00     19939
   macro avg       0.97      0.88      0.92     19939
weighted avg       1.00      1.00      1.00     19939


🔥 Top Features:
url_len        0.379228
ip_enc         0.201146
size           0.113276
hour           0.111973
url_depth      0.099530
is_image       0.040572
is_html        0.038320
weekday        0.007189
day            0.006873
method_HEAD    0.001894
dtype: float64

💾 Model saved → D:\python\Projects\Webloganalyzer\models\error_model.pkl
